# PATSTAT Disruption Trend — year-by-year disruption (CD) and citer profile (F/E/G) per application

The twin of `PatentView/notebook/patent_disruption_trend.ipynb`, anchored at the **filing year**: for every
application with at least one citer, one row per application × year in which anything changed, carrying

- `ni_new, nj_new, nk_new` — citing / co-citing applications that appeared **in that year**
  ($n_i$: cites the application but none of its references; $n_j$: cites it **and** ≥ 1 of its references;
  $n_k$: cites ≥ 1 of its references but not the application);
- `ni, nj, nk` — the **cumulative** counts from the filing year through that year;
- `CD` — $\dfrac{n_i-n_j}{n_i+n_j+n_k}$ on the cumulative counts, i.e. the disruption index with a window of
  exactly `yrs_since_filing` years;
- `F, E, G` — the Foundational / Extensional / Generalizational shares of the citers that have arrived by
  that year, NaN until the first citer.

This is the time axis that `patstat_disruption.ipynb` collapses into four windows: the row at
`yrs_since_filing = w` (or the last row before it) **is** `CD_w` / `F_w` / `E_w` / `G_w`, and the last row is
the `_all` value. Section 3 checks that identity on **every** focal application against
`patstat_disruption.parquet` (matched on `appln_id`, not on row position).

## Input
```
PATSTAT/cache/patstat_csr.npz            # the de-duplicated application graph patstat_disruption built (ps.EDGE_WHERE)
PATSTAT/output/patstat_disruption.parquet  # per-window CD/F/E/G/ni/nj/nk: the check target
```
The kernel is the PatentView / OpenAlex `trend_kernel`, verbatim.

## Output
`PATSTAT/output/patstat_disruption_trend.parquet` —
`appln_id, filing_year, cite_year, yrs_since_filing, ni_new, nj_new, nk_new, ni, nj, nk, CD, F, E, G`.

`PATSTAT/output/patstat_disruption_trend_summary.parquet` — `filing_year, yrs_since_filing, n_<m>, <m>_mean` for
`m` in CD/F/E/G: mean cumulative outcomes by cohort and age.

> **Clock switch.** This text describes the default filing clock (`output/`). With `NB_PS_CLOCK=grant` the same notebook runs on the grant clock, PatentView's: years and ages are grant years (first publication with `publn_first_grant = 'Y'`), only applications with a grant year are in the universe (both ends of every edge), year / age columns are named `grant_year` / `yrs_since_grant`, and the outputs go to `PATSTAT/output_grant/` (executed copy: `notebook/grant/`).

> **Family unit.** With `NB_PS_UNIT=family` the same notebook runs with the DOCDB family as the unit (`docdb_family_id`), dated by the family's earliest priority year (`priority_year`, `yrs_since_priority`): citations are distinct family-to-family pairs with within-family citations dropped, classification comes from the earliest member that has one, inventors and their countries from one representative member (the one with the most located inventors), and the outputs go to `PATSTAT/output_family/` (executed copy: `notebook/family/`).


In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
import numba
OUT_FP  = ps.out('patstat_disruption_trend.parquet')
SUM_FP  = ps.out('patstat_disruption_trend_summary.parquet')
DISR_FP = ps.out('patstat_disruption.parquet')
CSR_NPZ = ps.cache('patstat_csr.npz')
BATCH   = 1_000_000                               # focal applications per kernel call / write
NT = int(os.environ.get('NB_WORKERS', numba.config.NUMBA_NUM_THREADS))
numba.set_num_threads(min(NT, numba.config.NUMBA_NUM_THREADS))
print('numba threads:', numba.get_num_threads())
ps.preflight('patstat_disruption_trend')
assert os.path.exists(CSR_NPZ), f'{CSR_NPZ} missing -- run patstat_disruption first'
print('output ->', OUT_FP)

numba threads: 16
dump   : /project/jevans/PATSTAT/unzipped_data   (65 zip parts, 11 tables registered)
raw    : /project/jevans/Dawoon/Science of Science/PATSTAT/raw
output : /project/jevans/Dawoon/Science of Science/PATSTAT/output

  patstat_disruption_trend      OK
output -> /project/jevans/Dawoon/Science of Science/PATSTAT/output/patstat_disruption_trend.parquet


## 1. The graph (cached CSR from `patstat_disruption`)

In [2]:
%%time
z = np.load(CSR_NPZ)
out_ptr, out_idx, in_ptr, in_idx, year, uni = z['out_ptr'], z['out_idx'], z['in_ptr'], z['in_idx'], z['year'], z['uni']
del z; gc.collect()
n = len(uni); YMAX = int(year.max())
print(f'CSR: {n:,} applications, {len(out_idx):,} edges | {ps.CLOCK} years {int(year.min())}..{YMAX}')

CSR: 54,330,212 applications, 335,409,894 edges | filing years 1900..2023


## 2. Numba engine (verbatim from `patent_disruption_trend`)

In [3]:
from numba import njit, prange

@njit(inline='always')
def _bfind(arr, x):
    lo = 0; hi = len(arr)
    while lo < hi:
        mid = (lo + hi) >> 1
        if arr[mid] < x: lo = mid + 1
        else: hi = mid
    return lo < len(arr) and arr[lo] == x

@njit(parallel=True)
def trend_kernel(focal, off, out_ptr, out_idx, in_ptr, in_idx, year,
                 ni, nj, nk, NI, NJ, NK, CD, Ff, Ef, Gf):
    """Per focal document, per year since publication: new and cumulative n_i / n_j / n_k, CD,
    and the cumulative citer profile F / E / G.

    Dense layout: focal t owns slots off[t] .. off[t+1]-1, slot off[t]+d being year yF+d.
    ni/nj/nk receive the yearly increments, NI/NJ/NK the running totals, CD the running index,
    Ff/Ef/Gf the shares of the citers of age <= d that are Foundational / Extensional /
    Generalizational at age d. Slots of different focal documents never overlap, so the prange
    writes are race-free.

    Same definitions as the per-window engine (paper_disruption / patent_disruption):
      n_j(y) = citers published in year y that cite >= 1 reference of the focal;
      n_i(y) = the other citers of year y;
      n_k(y) = |B_y| - n_j(y), B_y = distinct documents of year y citing >= 1 reference of the
               focal, the focal itself excluded;
      citer c: up = |refs(F) & refs(c)|, down(d) = |{citers of F of age <= d} & refs(c)|;
               up > down -> E, down > up -> F, up = down = 0 -> G, up = down > 0 -> half F, half E.
    Summing over years 0..w reproduces ni_w / nj_w / nk_w / F_w / E_w / G_w exactly, because a
    window in the per-window engine is precisely a prefix of years. `down` grows with the age,
    which is why the profile is re-cut at every age instead of once.
    """
    for t in prange(len(focal)):
        F = focal[t]; yF = year[F]; b0 = off[t]; b1 = off[t + 1]; span = b1 - b0
        a0 = in_ptr[F]; a1 = in_ptr[F + 1]
        R = out_idx[out_ptr[F]:out_ptr[F + 1]]
        Rs = np.sort(R); As = np.sort(in_idx[a0:a1])
        # per-age citer-profile counters; a citer of age dc is present at every age >= dc
        Nc = np.zeros(span, np.int64); cF = np.zeros(span, np.int64); cE = np.zeros(span, np.int64)
        cG = np.zeros(span, np.int64); cT = np.zeros(span, np.int64)
        maxdeg = 0
        for ci in range(a0, a1):
            c = in_idx[ci]; deg = out_ptr[c + 1] - out_ptr[c]
            if deg > maxdeg: maxdeg = deg
        dds = np.empty(maxdeg, np.int64)
        for ci in range(a0, a1):
            c = in_idx[ci]; dc = year[c] - yF
            if dc < 0:
                continue
            up = 0; nd = 0
            for ri in range(out_ptr[c], out_ptr[c + 1]):
                d = out_idx[ri]
                if _bfind(Rs, d): up += 1
                if _bfind(As, d):
                    dd = year[d] - yF
                    if dd >= 0:
                        dds[nd] = dd; nd += 1
            if up > 0: nj[b0 + dc] += 1
            else:      ni[b0 + dc] += 1
            dsort = np.sort(dds[:nd]); p = 0
            for tt in range(dc, span):
                while p < nd and dsort[p] <= tt: p += 1     # down(tt) = co-cited citers of age <= tt
                Nc[tt] += 1
                if up > p:   cE[tt] += 1
                elif p > up: cF[tt] += 1
                elif up > 0: cT[tt] += 1
                else:        cG[tt] += 1
        # B: distinct citers of the focal's references (focal excluded), by year
        totB = 0
        for ri in range(len(R)):
            r = R[ri]; totB += in_ptr[r + 1] - in_ptr[r]
        if totB > 0:
            buf = np.empty(totB, np.int32); p = 0
            for ri in range(len(R)):
                r = R[ri]
                for j in range(in_ptr[r], in_ptr[r + 1]):
                    buf[p] = in_idx[j]; p += 1
            buf.sort(); prev = np.int32(-1)
            for ii in range(totB):
                b = buf[ii]
                if b == prev or b == F:
                    continue
                prev = b; dd = year[b] - yF
                if dd >= 0:
                    nk[b0 + dd] += 1
        # nk holds |B_y| so far; turn it into n_k(y) = |B_y| - n_j(y), then cumulate
        si = 0; sj = 0; sk = 0
        for q in range(span):
            p = b0 + q
            nk[p] -= nj[p]
            si += ni[p]; sj += nj[p]; sk += nk[p]
            NI[p] = si; NJ[p] = sj; NK[p] = sk
            den = si + sj + sk
            CD[p] = (si - sj) / den if den > 0 else np.nan
            if Nc[q] > 0:
                Ff[p] = (cF[q] + 0.5 * cT[q]) / Nc[q]
                Ef[p] = (cE[q] + 0.5 * cT[q]) / Nc[q]
                Gf[p] = cG[q] / Nc[q]
            else:
                Ff[p] = np.nan; Ef[p] = np.nan; Gf[p] = np.nan
print('engine ready')

engine ready


## 3. Run over all focal applications in batches, stream to parquet, check every one against `patstat_disruption.parquet`

In [4]:
%%time
focal_all = np.flatnonzero(in_ptr[1:] - in_ptr[:-1] > 0).astype(np.int64)
print(f'focal applications with >= 1 citer: {len(focal_all):,}  (batches of {BATCH:,})')
YMIN = int(year.min()); NY = YMAX - YMIN + 1
OUTC = ['CD', 'F', 'E', 'G']; CNTS = ['ni', 'nj', 'nk']
SCHEMA = pa.schema([(ps.KEY, pa.int64()), (ps.YEAR_COL, pa.int32()), ('cite_year', pa.int32()),
                    (ps.AGE_COL, pa.int32()),
                    ('ni_new', pa.int32()), ('nj_new', pa.int32()), ('nk_new', pa.int32()),
                    ('ni', pa.int32()), ('nj', pa.int32()), ('nk', pa.int32()),
                    ('CD', pa.float32()), ('F', pa.float32()), ('E', pa.float32()), ('G', pa.float32())])
sum_o = {m: np.zeros(NY * NY, np.float64) for m in OUTC}; cnt_o = {m: np.zeros(NY * NY, np.int64) for m in OUTC}

# the per-window table, aligned to the CSR's code order through appln_id (its rows need not be in that order)
WIN = {'_3': 3, '_5': 5, '_10': 10, '_all': None}
_dt = pq.read_table(DISR_FP, columns=[ps.KEY] + [m + s_ for s_ in WIN for m in OUTC + CNTS])
ids = _dt.column(ps.KEY).to_numpy()
order = np.argsort(ids, kind='stable'); pos = order[np.searchsorted(ids, uni, sorter=order)]
assert np.array_equal(ids[pos], uni), 'patstat_disruption.parquet does not cover the CSR nodes'
DIS = {}
for s_ in WIN:
    for m in OUTC:
        DIS[m + s_] = _dt.column(m + s_).to_numpy(zero_copy_only=False).astype(np.float32)[pos]
    for m in CNTS:                     # -1 means "nothing in the window"
        v = _dt.column(m + s_).to_numpy(zero_copy_only=False).astype(np.float64)[pos]
        DIS[m + s_] = np.where(np.isfinite(v), v, -1).astype(np.int64)
del _dt, ids, order, pos; gc.collect()
print(f'per-window table loaded for the check: {len(DIS["CD_3"]):,} rows x {len(DIS)} columns')
mism = {s_: {m: 0 for m in OUTC + CNTS + ['undefined']} for s_ in WIN}
ncmp = {s_: 0 for s_ in WIN}

TMP = OUT_FP + '.tmp'
writer = pq.ParquetWriter(TMP, SCHEMA, compression='zstd')
rows_total = 0; docs_total = 0; t0 = time.time()
nb = (len(focal_all) + BATCH - 1) // BATCH
for bi, s in enumerate(range(0, len(focal_all), BATCH)):
    focal = focal_all[s:s + BATCH]
    yF = year[focal].astype(np.int64)
    span = YMAX - yF + 1
    off = np.zeros(len(focal) + 1, np.int64); np.cumsum(span, out=off[1:])
    T = int(off[-1])
    ni = np.zeros(T, np.int32); nj = np.zeros(T, np.int32); nk = np.zeros(T, np.int32)
    NI = np.empty(T, np.int32); NJ = np.empty(T, np.int32); NK = np.empty(T, np.int32)
    CD = np.empty(T, np.float32); Ff = np.empty(T, np.float32); Ef = np.empty(T, np.float32); Gf = np.empty(T, np.float32)
    trend_kernel(focal, off, out_ptr, out_idx, in_ptr, in_idx, year, ni, nj, nk, NI, NJ, NK, CD, Ff, Ef, Gf)

    # exhaustive check: the cumulative slot at age w IS the per-window value at w
    OUT_ARR = {'CD': CD, 'F': Ff, 'E': Ef, 'G': Gf, 'ni': NI, 'nj': NJ, 'nk': NK}
    for s_, w in WIN.items():
        slot = off[:-1] + (span - 1 if w is None else np.minimum(w, span - 1))
        undefined = (NI[slot] == 0) & (NJ[slot] == 0) & (NK[slot] == 0)
        ref_undef = DIS['ni' + s_][focal] == -1
        mism[s_]['undefined'] += int((undefined != ref_undef).sum())
        ok = ~undefined & ~ref_undef
        for m in CNTS:
            mism[s_][m] += int((OUT_ARR[m][slot][ok] != DIS[m + s_][focal][ok]).sum())
        for m in OUTC:
            mism[s_][m] += int((~np.isclose(OUT_ARR[m][slot][ok], DIS[m + s_][focal][ok], atol=1e-6, equal_nan=True)).sum())
        ncmp[s_] += int(ok.sum())

    ridx = np.flatnonzero((ni != 0) | (nj != 0) | (nk != 0))
    fpos = np.searchsorted(off, ridx, side='right') - 1
    yrs = (ridx - off[fpos]).astype(np.int32)
    fy = yF[fpos].astype(np.int32)
    tbl = pa.table({ps.KEY: uni[focal][fpos], ps.YEAR_COL: fy, 'cite_year': fy + yrs, ps.AGE_COL: yrs,
                    'ni_new': ni[ridx], 'nj_new': nj[ridx], 'nk_new': nk[ridx],
                    'ni': NI[ridx], 'nj': NJ[ridx], 'nk': NK[ridx],
                    'CD': CD[ridx], 'F': Ff[ridx], 'E': Ef[ridx], 'G': Gf[ridx]}, schema=SCHEMA)
    writer.write_table(tbl)
    key_all = (fy.astype(np.int64) - YMIN) * NY + yrs
    for m in OUTC:
        v = OUT_ARR[m][ridx]; fin = np.isfinite(v)
        sum_o[m] += np.bincount(key_all[fin], weights=v[fin].astype(np.float64), minlength=NY * NY)
        cnt_o[m] += np.bincount(key_all[fin], minlength=NY * NY)
    rows_total += len(ridx); docs_total += int((np.bincount(fpos, minlength=len(focal)) > 0).sum())
    print(f'[{time.time()-t0:6.0f}s] batch {bi+1:>3}/{nb}: {len(focal):,} focal -> {len(ridx):,} rows  (cum {rows_total:,})', flush=True)
    del ni, nj, nk, NI, NJ, NK, CD, Ff, Ef, Gf, OUT_ARR, ridx, fpos, yrs, fy, tbl, key_all, v, fin; gc.collect()
writer.close()
os.replace(TMP, OUT_FP)
del DIS; gc.collect()
print(f'WROTE {OUT_FP}  ({rows_total:,} rows, {os.path.getsize(OUT_FP)/1e9:.2f} GB)')
print(f'  applications covered: {docs_total:,} | cite_year {YMIN}..{YMAX}')

focal applications with >= 1 citer: 40,901,611  (batches of 1,000,000)
per-window table loaded for the check: 54,330,212 rows x 28 columns
[    14s] batch   1/41: 1,000,000 focal -> 6,202,188 rows  (cum 6,202,188)
[    17s] batch   2/41: 1,000,000 focal -> 4,802,301 rows  (cum 11,004,489)
[    21s] batch   3/41: 1,000,000 focal -> 7,050,871 rows  (cum 18,055,360)
[    31s] batch   4/41: 1,000,000 focal -> 11,018,785 rows  (cum 29,074,145)
[    40s] batch   5/41: 1,000,000 focal -> 12,184,664 rows  (cum 41,258,809)
[    46s] batch   6/41: 1,000,000 focal -> 6,115,237 rows  (cum 47,374,046)
[    55s] batch   7/41: 1,000,000 focal -> 6,144,755 rows  (cum 53,518,801)
[    58s] batch   8/41: 1,000,000 focal -> 3,224,882 rows  (cum 56,743,683)
[    61s] batch   9/41: 1,000,000 focal -> 3,443,100 rows  (cum 60,186,783)
[    64s] batch  10/41: 1,000,000 focal -> 3,415,972 rows  (cum 63,602,755)
[    68s] batch  11/41: 1,000,000 focal -> 3,543,106 rows  (cum 67,145,861)
[    71s] batch  12/41: 

In [5]:
print(f'{"window":<8}{"compared":>14}' + ''.join(f'{m:>10}' for m in OUTC + CNTS) + f'{"undefined":>12}')
print('-' * 92)
total_bad = 0
for s_ in WIN:
    print(f'{s_:<8}{ncmp[s_]:>14,}' + ''.join(f'{mism[s_][m]:>10,}' for m in OUTC + CNTS) + f'{mism[s_]["undefined"]:>12,}')
    total_bad += sum(mism[s_].values())
assert total_bad == 0, 'trend disagrees with the per-window table'
print(f'\n  cumulative trend == per-window CD / F / E / G / ni / nj / nk at every window, on all {len(focal_all):,} applications')

window        compared        CD         F         E         G        ni        nj        nk   undefined
--------------------------------------------------------------------------------------------
_3          29,547,430         0         0         0         0         0         0         0           0
_5          32,017,122         0         0         0         0         0         0         0           0
_10         35,284,457         0         0         0         0         0         0         0           0
_all        40,901,611         0         0         0         0         0         0         0           0

  cumulative trend == per-window CD / F / E / G / ni / nj / nk at every window, on all 40,901,611 applications


## 4. Summary — mean cumulative CD / F / E / G by age, pooled and by cohort; head of the output

In [6]:
cells = np.flatnonzero(sum(cnt_o.values()))
py, yy = np.divmod(cells, NY)
summ = pd.DataFrame({ps.YEAR_COL: py + YMIN, ps.AGE_COL: yy})
with np.errstate(invalid='ignore', divide='ignore'):
    for m in OUTC:
        summ[f'n_{m}'] = cnt_o[m][cells]
        summ[f'{m}_mean'] = sum_o[m][cells] / cnt_o[m][cells]
summ.to_parquet(SUM_FP, index=False)
print(f'WROTE {SUM_FP}  ({len(summ):,} ({ps.YEAR_COL}, {ps.AGE_COL}) cells)')

S = {m: sum_o[m].reshape(NY, NY) for m in OUTC}; C = {m: cnt_o[m].reshape(NY, NY) for m in OUTC}
YRS = [y for y in [0, 1, 2, 3, 5, 10, 15, 20, 30] if y < NY]
pooled = pd.DataFrame({ps.AGE_COL: YRS, 'n_CD': [int(C['CD'][:, y].sum()) for y in YRS], 'n_FEG': [int(C['F'][:, y].sum()) for y in YRS],
                       **{f'{m}_mean': [S[m][:, y].sum() / max(C[m][:, y].sum(), 1) for y in YRS] for m in OUTC}})
display(pooled.round(4))
rows = []
for lo, hi in ((1960, 1969), (1970, 1979), (1980, 1989), (1990, 1999), (2000, 2009), (2010, 2019)):
    if hi < YMIN or lo > YMAX:
        continue
    r = slice(max(lo, YMIN) - YMIN, min(hi, YMAX) - YMIN + 1)
    rows.append({'cohort': f'{lo}-{hi}', 'n_rows': int(C['CD'][r].sum()),
                 **{f'{m}@{y}': (S[m][r, y].sum() / C[m][r, y].sum() if y < NY and C[m][r, y].sum() else np.nan)
                    for m in OUTC for y in (1, 3, 5, 10, 20)}})
COH = pd.DataFrame(rows).set_index('cohort')
for m in OUTC:
    print(f'\n{m} by filing cohort at fixed ages:')
    display(COH[[f'{m}@{y}' for y in (1, 3, 5, 10, 20)]].round(4))
head = pq.ParquetFile(OUT_FP).read_row_group(0).to_pandas()
with pd.option_context('display.width', 250, 'display.max_columns', 20):
    display(head.head(12))

WROTE /project/jevans/Dawoon/Science of Science/PATSTAT/output/patstat_disruption_trend_summary.parquet  (7,319 (filing_year, yrs_since_filing) cells)


,yrs_since_filing,n_CD,n_FEG,CD_mean,F_mean,E_mean,G_mean
0,0,22482874,4019535,0.0358,0.0218,0.3300,0.6482
1,1,23068963,10472532,0.0980,0.0221,0.2517,0.7262
2,2,23457592,15526768,0.1499,0.0273,0.2177,0.7549
3,3,22784791,17654446,0.1708,0.0350,0.2047,0.7603
4,5,19731713,17223417,0.1762,0.0546,0.1980,0.7474
5,10,12737998,12063535,0.1948,0.1017,0.1906,0.7077
6,15,8216752,7966520,0.2035,0.1382,0.1858,0.6760
7,20,5036114,4925050,0.2204,0.1614,0.1753,0.6632
8,30,2095673,2064797,0.2399,0.1775,0.1521,0.6704



CD by filing cohort at fixed ages:


,CD@1,CD@3,CD@5,CD@10,CD@20
cohort,,,,,
1960-1969,0.0630,0.1607,0.1901,0.2303,0.2772
1970-1979,0.1082,0.2753,0.2993,0.3029,0.2476
1980-1989,0.0947,0.2273,0.2192,0.1962,0.2099
1990-1999,0.0589,0.1840,0.2037,0.2259,0.2209
2000-2009,0.0652,0.1827,0.1940,0.1930,0.1392
2010-2019,0.1012,0.1476,0.1343,0.1108,NaN



F by filing cohort at fixed ages:


...


E by filing cohort at fixed ages:


,E@1,E@3,E@5,E@10,E@20
cohort,,,,,
1960-1969,0.2485,0.2084,0.1923,0.1670,0.1414
1970-1979,0.2388,0.1889,0.1804,0.1692,0.1647
1980-1989,0.2638,0.2107,0.2124,0.2050,0.1753
1990-1999,0.3264,0.2531,0.2362,0.2061,0.1807
2000-2009,0.3358,0.2332,0.2144,0.1903,0.1965
2010-2019,0.2252,0.1805,0.1762,0.1817,NaN



G by filing cohort at fixed ages:


,G@1,G@3,G@5,G@10,G@20
cohort,,,,,
1960-1969,0.7490,0.7811,0.7836,0.7825,0.7737
1970-1979,0.7575,0.7971,0.7897,0.7684,0.7077
1980-1989,0.7291,0.7724,0.7511,0.7092,0.6688
1990-1999,0.6653,0.7220,0.7117,0.6815,0.6386
2000-2009,0.6320,0.7236,0.7221,0.6997,0.6091
2010-2019,0.7509,0.7824,0.7682,0.7168,NaN


,appln_id,filing_year,cite_year,yrs_since_filing,ni_new,nj_new,nk_new,ni,nj,nk,CD,F,E,G
0,1,2000,2000,0,3,0,3,3,0,3,0.500000,0.000000,0.0,1.000000
1,1,2000,2001,1,2,0,3,5,0,6,0.454545,0.000000,0.0,1.000000
2,1,2000,2002,2,3,0,3,8,0,9,0.470588,0.000000,0.0,1.000000
3,1,2000,2003,3,3,0,3,11,0,12,0.478261,0.000000,0.0,1.000000
4,1,2000,2004,4,2,0,3,13,0,15,0.464286,0.000000,0.0,1.000000
5,1,2000,2005,5,1,0,0,14,0,15,0.482759,0.000000,0.0,1.000000
6,1,2000,2006,6,1,0,1,15,0,16,0.483871,0.066667,0.0,0.933333
7,1,2000,2007,7,1,0,2,16,0,18,0.470588,0.062500,0.0,0.937500
8,1,2000,2009,9,1,0,0,17,0,18,0.485714,0.058824,0.0,0.941176
9,1,2000,2011,11,1,0,1,18,0,19,0.486486,0.055556,0.0,0.944444
